# 01 · Chuẩn bị dữ liệu bằng code chính thức (runtime **CPU**)

Nhánh `explore/official-code`. Mục tiêu: chạy **đúng pipeline tiền xử lý của tác giả** (`third_party/vlm2rec_anon/datasets/*`) cho một dataset Amazon, và đo thống kê để so với bảng của paper.

* **Runtime:** `Runtime → Change runtime type → CPU`. Bước này không cần GPU, nên không tốn compute unit.
* **Kết quả lưu vào Drive** (`MyDrive/VLM2Rec_work/`):
  * `data/<DATASET>_processed.tar.gz` (dùng cho notebook 02)
  * `audit_results/prep_report_<DATASET>.json`
* **Thời gian ước tính (Beauty):** 15–30 phút, chủ yếu là tải metadata và ~10k ảnh.
* Code gốc **không hỗ trợ MovieLens-1M** (không có script poster TMDB) → chỉ chọn được 4 dataset Amazon.

In [ ]:
REPO = "TrNguyenMQuan/VLM2Rec"
BRANCH = "explore/official-code"
DATASET = "Beauty"        # Beauty | Toys_and_Games | Sports | Clothing
WORKERS = 16              # luồng tải song song; 1 = giống hệt code gốc (tuần tự, chậm hơn nhiều)
DRIVE_ROOT = "/content/drive/MyDrive/VLM2Rec_work"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, shutil, subprocess

def clone_repo():
    """Public repo: clone directly. Private repo: add a Colab secret GITHUB_TOKEN (read access)."""
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None
    dst = "/content/VLM2Rec"
    if os.path.exists(f"{dst}/.git"):
        subprocess.run(["git", "-C", dst, "pull", "--ff-only"], check=True)
    else:
        url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
        subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", url, dst], check=True)
    print(subprocess.run(["git", "-C", dst, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

clone_repo()
TOOLS = "/content/VLM2Rec/tools/official"
WORK = "/content/work/official"          # working copy of the official code (repo copy stays untouched)
if not os.path.exists(WORK):
    shutil.copytree("/content/VLM2Rec/third_party/vlm2rec_anon", WORK)
AUDIT = f"{DRIVE_ROOT}/audit_results"
for p in (DRIVE_ROOT, AUDIT, f"{DRIVE_ROOT}/data", f"{DRIVE_ROOT}/checkpoints", f"{DRIVE_ROOT}/logs"):
    os.makedirs(p, exist_ok=True)
print("WORK =", WORK, "| AUDIT =", AUDIT)

In [ ]:
# preprocess_data.py dùng groupby().apply(tail) — hành vi này đổi ở pandas 3, nên giữ pandas 2.x
!pip install -q "pandas>=2.2,<3" tqdm requests pillow
!python -c "import pandas, requests, PIL; print('pandas', pandas.__version__, '| requests', requests.__version__, '| Pillow', PIL.__version__)"

## Chạy tiền xử lý chính thức
`prep_amazon.py` gọi trực tiếp các hàm của tác giả: `load_metadata_df`, `load_dataset_df`, `preprocess_sequential(max_seq_len=10, reapply_core_after_trim=True)`, `filtering_csv_sep`, `build_user_item_json` và logic tải ảnh. Chỉ phần gọi mạng được chạy song song. Ngoài ra, script tính thêm **Checkpoint A** (5-core + join metadata, *trước* khi cắt chuỗi) để so với bảng thống kê trong paper.

In [ ]:
!python {TOOLS}/prep_amazon.py --official_dir {WORK} --dataset {DATASET} --workers {WORKERS} --report {AUDIT}/prep_report_{DATASET}.json

In [ ]:
import json, sys
sys.path.insert(0, TOOLS)
from paper_targets import DATA_STATS
r = json.load(open(f"{AUDIT}/prep_report_{DATASET}.json"))
paper = DATA_STATS[DATASET]
print(f"{'stage':45s} {'users':>8s} {'items':>8s} {'inter':>9s} {'avg':>6s}")
print(f"{'paper table':45s} {paper['users']:8d} {paper['items']:8d} {paper['interactions']:9d} {paper['avg_len']:6}")
for k in ("raw_reviews_5core_file", "after_5core", "checkpoint_A", "official_output"):
    s = r[k]; print(f"{k:45s} {s['users']:8d} {s['items']:8d} {s['interactions']:9d} {s['avg_len']:6}")
print("image URL check:", r.get("image_url_check"))
print("images:", {k: r["images"][k] for k in ("n_items_in_meta", "downloaded", "failed", "missing_after")})

**Cần quan sát:**
1. `checkpoint_A` có khớp `paper table` không? Nếu khớp → bảng thống kê trong paper ứng với dữ liệu **chưa cắt**.
2. `official_output` (sau khi cắt còn 10 item cuối, lọc ảnh, áp 5-core lại) khác bao nhiêu → đây là dữ liệu mà code gốc thực sự dùng để train/eval.
3. `missing_after` phải bằng 0. Nếu > 0, chạy lại cell trên vì script bỏ qua ảnh đã tải. Trainer của tác giả sẽ crash nếu thiếu ảnh.

In [ ]:
# Lưu dữ liệu đã xử lý lên Drive để notebook 02 dùng
tarball = f"{DRIVE_ROOT}/data/{DATASET}_processed.tar.gz"
!tar czf {tarball} -C {WORK}/datasets processed/{DATASET}.txt processed/{DATASET}_item_meta.csv processed/{DATASET}_meta.json processed/{DATASET}_image_pool
!ls -lh {tarball}

✅ Xong. Mở **`02_official_stage1_stage2_gpu.ipynb`** với runtime GPU (A100 khuyến nghị, hoặc L4).